In [4]:
!pip install ucimlrepo
!pip install joblib
!pip install scipy

In [1]:
import time
import warnings
from ucimlrepo import fetch_ucirepo
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.integrate import quad, dblquad
from scipy.stats import multivariate_normal
from collections import defaultdict
import traceback

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Set random seed for reproducibility IN EACH RUN if needed for true averaging,
# but fixed seed is better for comparing parameter sets. Keep fixed for now.
np.random.seed(42)

# --- Data Loading and Preprocessing ---
#print("Loading datasets...")
iris = fetch_ucirepo(id=53)
wine = fetch_ucirepo(id=109)


iris_label_map = {'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}
iris_targets = np.array([iris_label_map[label] for label in iris.data.targets.values.ravel()])



wine_targets = wine.data.targets.values.ravel().astype(int)


#print("Standardizing data...")
scaler = StandardScaler()
datasets = {
    "Iris": scaler.fit_transform(iris.data.features.values),
    "Wine": scaler.fit_transform(wine.data.features.values),

}
true_labels = {
    "Iris": iris_targets,
    "Wine": wine_targets,
    # "Glass": glass_targets,
    # "Ecoli": ecoli_targets
}
k_values = {name: len(np.unique(targets)) for name, targets in true_labels.items()}



# --- Core Classes (Entry, Dataset, Classification) ---
# Unchanged from previous version
class Entry:
    def __init__(self, id_, class_label_id, values, u_values):
        self.id_ = id_
        self.class_label_id = int(class_label_id)
        self.values = np.array(values)
        self.u_values = u_values
    def get_id(self): return self.id_
    def get_values(self): return self.values
    def set_pdf(self, pdf, index):
        if not isinstance(self.u_values, list): self.u_values = [None] * len(self.values)
        if index < len(self.u_values): self.u_values[index] = pdf
        else: raise IndexError(f"Index {index} out of bounds")
    def get_random_sample(self):
        if self.u_values is None: return self.values
        if isinstance(self.u_values, list):
            sample = [pdf.get_random_sample() if pdf is not None else self.values[idx] for idx, pdf in enumerate(self.u_values)]
            return np.array(sample)
        else: return np.array(self.u_values.get_random_sample())

class Dataset:
    def __init__(self, entries): self.entries = entries
    def get_size(self): return len(self.entries)
    def get_entry(self, i): return self.entries[i]

class Classification:
    def __init__(self, objects): self.labels = [int(obj.class_label_id) for obj in objects]
    def get_label(self, i): return self.labels[i]

# --- PDF Classes (MultivariateUniform, MultivariateGaussian, Binomial) ---
# Unchanged from previous version
class MultivariateUniform:
    def __init__(self, min_values, max_values, n_samples=None):
        self.min_values = np.array(min_values)
        self.max_values = np.array(max_values)
        self.dim = len(min_values)
        self.volume = np.prod([(max_val - min_val) if (max_val > min_val) else 1e-9 for min_val, max_val in zip(self.min_values, self.max_values)])
        self.pdf_value = 1.0 / self.volume if self.volume > 1e-12 else 0
    def pdf(self, x):
        x = np.array(x)
        if x.shape != self.min_values.shape: return 0.0 # Handle shape mismatch gracefully
        if self.volume <= 1e-12: return 0.0
        return self.pdf_value if np.all((x >= self.min_values) & (x <= self.max_values)) else 0.0
    def get_random_sample(self): return np.random.uniform(self.min_values, self.max_values, size=self.dim)

class MultivariateGaussian:
    def __init__(self, min_values, max_values, n_samples=None, rho=0.05):
        self.min_values = np.array(min_values)
        self.max_values = np.array(max_values)
        self.rho = max(1e-6, min(1.0, rho))
        self.mu = (self.min_values + self.max_values) / 2.0
        self.dim = len(min_values)
        variances = [(max_val - min_val)**2 / 12.0 if max_val > min_val else 1e-6 for min_val, max_val in zip(self.min_values, self.max_values)]
        self.sigma = np.diag(variances) * self.rho
        self.sigma += np.eye(self.dim) * 1e-9 # Add epsilon for stability
        try:
            # Test Cholesky decomposition to check positive definiteness
            np.linalg.cholesky(self.sigma)
            self._scipy_pdf_func = multivariate_normal(mean=self.mu, cov=self.sigma, allow_singular=False)
        except np.linalg.LinAlgError:
            # print(f"Warning: Gaussian Covariance adjustment needed. Rho={self.rho}, Mu={self.mu.mean():.2f}")
            diag_sigma = np.maximum(np.diag(self.sigma), 1e-6) # Ensure diagonal positive
            self.sigma = np.diag(diag_sigma)
            try:
                 np.linalg.cholesky(self.sigma)
                 self._scipy_pdf_func = multivariate_normal(mean=self.mu, cov=self.sigma, allow_singular=False)
            except np.linalg.LinAlgError:
                 # print(f"FATAL: Gaussian Covariance fallback. Rho={self.rho}")
                 self.sigma = np.eye(self.dim) * 0.1 # Fallback covariance
                 self._scipy_pdf_func = multivariate_normal(mean=self.mu, cov=self.sigma, allow_singular=False)

    def pdf(self, x):
        x = np.array(x)
        if x.shape != self.mu.shape: return 0.0 # Handle shape mismatch
        try: return self._scipy_pdf_func.pdf(x)
        except: return 0.0 # Catch any PDF calculation errors
    def get_random_sample(self):
        try: return np.random.multivariate_normal(self.mu, self.sigma)
        except np.linalg.LinAlgError: return self.mu # Return mean if sampling fails

class Binomial: # Represents Univariate Uniform
    def __init__(self, lower_bound, upper_bound, value=None, n_samples=None):
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound
        if self.upper_bound <= self.lower_bound: self.upper_bound = self.lower_bound + 1e-9
        self.range = self.upper_bound - self.lower_bound
        self.pdf_value = 1.0 / self.range if self.range > 1e-12 else 0.0
    def pdf(self, x):
         if self.range <= 1e-12: return 0.0
         return self.pdf_value if self.lower_bound <= x <= self.upper_bound else 0.0
    def get_random_sample(self): return np.random.uniform(self.lower_bound, self.upper_bound)


# --- Analytical Distance Calculation ---
# Unchanged from previous version (uses MC=10k samples)
class AnalyticalDistanceCalculator:
    def __init__(self, dataset):
        self.dataset = dataset
        self.dim = len(dataset.get_entry(0).get_values()) if dataset.get_size() > 0 else 0
        self._mc_fallback_count = 0
        self._total_distance_calcs = 0
        self._int_opts = {'epsabs':1.49e-06, 'epsrel':1.49e-06}
        self._gaussian_bound_stddevs = 6
    def compute_distance_matrix(self, dpm):
        n = self.dataset.get_size()
        if n == 0: return np.array([])
        distance_matrix = np.zeros((n, n))
        #print(f"Computing {n}x{n} distance matrix (dim={self.dim})...")
        start_time = time.time()
        self._mc_fallback_count = 0; self._total_distance_calcs = 0
        for i in range(n):
            e1 = self.dataset.get_entry(i)
            row_distances = []
            for j in range(i + 1):
                if i == j: distance = 0.0
                else:
                    e2 = self.dataset.get_entry(j)
                    try: distance = self._compute_pairwise_distance(e1, e2)
                    except Exception as e:
                        # print(f"Dist calc error pair ({i},{j}): {e}. Using center dist.")
                        distance = max(1e-6, np.linalg.norm(e1.get_values() - e2.get_values()))
                if not np.isfinite(distance) or distance < 0:
                     distance = max(1e-6, np.linalg.norm(e1.get_values() - e2.get_values()))
                distance_matrix[i, j] = distance; distance_matrix[j, i] = distance
                row_distances.append(distance)
            dpm.write_data_row(row_distances[:i+1])
            if (i + 1) % (n // 10 if n > 50 else 50) == 0:
                 elapsed = time.time() - start_time
                 rate = (i + 1) / elapsed if elapsed > 0 else 0
                 # print(f"  Computed {i+1}/{n} rows... ({elapsed:.2f}s, {rate:.1f} rows/s)")

        #print(f"Distance matrix computation finished in {time.time() - start_time:.2f}s")
        if self._total_distance_calcs > 0:
            mc_percentage = (self._mc_fallback_count / self._total_distance_calcs) * 100
            #print(f"  Monte Carlo fallback used for {self._mc_fallback_count}/{self._total_distance_calcs} ({mc_percentage:.2f}%) distances.")
        return distance_matrix

    def _compute_pairwise_distance(self, e1, e2):
        self._total_distance_calcs += 1
        u1 = e1.u_values; u2 = e2.u_values; expected_sq_dist = -1.0
        # Logic for choosing analytical, numerical, or MC based on PDF types
        if u1 is not None and u2 is not None:
            if isinstance(u1, MultivariateUniform) and isinstance(u2, MultivariateUniform): expected_sq_dist = self._uniform_uniform_sq_distance(u1, u2)
            elif isinstance(u1, MultivariateGaussian) and isinstance(u2, MultivariateGaussian): expected_sq_dist = self._gaussian_gaussian_sq_distance(u1, u2)
            elif isinstance(u1, list) and isinstance(u2, list): expected_sq_dist = self._univariate_list_sq_distance(u1, u2)
            else: expected_sq_dist = self._numerical_integration_sq_distance(e1, e2) # Fallback to MC usually
        elif u1 is not None and u2 is None: expected_sq_dist = self._pdf_point_sq_distance(u1, e2.get_values())
        elif u1 is None and u2 is not None: expected_sq_dist = self._pdf_point_sq_distance(u2, e1.get_values())
        else: expected_sq_dist = np.sum((e1.get_values() - e2.get_values())**2) # Crisp-Crisp
        if expected_sq_dist < -1e-9: raise ValueError(f"Sq dist calc failed pair ({e1.id_}, {e2.id_})")
        return np.sqrt(max(0, expected_sq_dist))

    def _uniform_uniform_sq_distance(self, u1: MultivariateUniform, u2: MultivariateUniform):
        if u1.dim != u2.dim: raise ValueError("Dim mismatch")
        ex1=(u1.min_values+u1.max_values)/2.0; ex2=(u2.min_values+u2.max_values)/2.0
        r1=u1.max_values-u1.min_values; r2=u2.max_values-u2.min_values
        v1=np.where(r1>1e-12,(r1**2)/12.0,0); v2=np.where(r2>1e-12,(r2**2)/12.0,0)
        return np.sum((ex1-ex2)**2+v1+v2)

    def _gaussian_gaussian_sq_distance(self, u1: MultivariateGaussian, u2: MultivariateGaussian):
        if u1.dim != u2.dim: raise ValueError("Dim mismatch")
        return np.sum((u1.mu-u2.mu)**2)+np.trace(u1.sigma)+np.trace(u2.sigma)

    def _univariate_list_sq_distance(self, u1_list, u2_list):
        if len(u1_list) != len(u2_list): raise ValueError("Dim mismatch")
        total_sq_dist = 0
        for d in range(len(u1_list)):
            p1=u1_list[d]; p2=u2_list[d]
            if isinstance(p1, Binomial) and isinstance(p2, Binomial):
                a1,b1=p1.lower_bound,p1.upper_bound; a2,b2=p2.lower_bound,p2.upper_bound
                r1=b1-a1; r2=b2-a2; ex1=(a1+b1)/2.0; ex2=(a2+b2)/2.0
                v1=(r1**2)/12.0 if r1>1e-12 else 0; v2=(r2**2)/12.0 if r2>1e-12 else 0
                total_sq_dist += (ex1-ex2)**2+v1+v2
            else: # Numerical integration needed if not both Binomial/UniUniform
                 integrand = lambda x, y: (x-y)**2 * p1.pdf(x) * p2.pdf(y)
                 b1=self._get_1d_bounds(p1); b2=self._get_1d_bounds(p2)
                 try: val, err = dblquad(integrand, b1[0],b1[1], lambda y: b2[0], lambda y: b2[1], **self._int_opts); total_sq_dist += val
                 except: # Fallback to 1D MC for this dimension
                     s1=[p1.get_random_sample() for _ in range(1000)]; s2=[p2.get_random_sample() for _ in range(1000)]
                     total_sq_dist += np.mean([(x-y)**2 for x,y in zip(s1,s2)])
        return total_sq_dist

    def _pdf_point_sq_distance(self, u_pdf, point_y):
        point_y = np.array(point_y)
        if isinstance(u_pdf, MultivariateGaussian): return np.sum((u_pdf.mu-point_y)**2)+np.trace(u_pdf.sigma)
        elif isinstance(u_pdf, MultivariateUniform):
            ex=(u_pdf.min_values+u_pdf.max_values)/2.0; r=u_pdf.max_values-u_pdf.min_values
            vx=np.where(r>1e-12,(r**2)/12.0,0); return np.sum((ex-point_y)**2+vx)
        elif isinstance(u_pdf, list):
             total_sq_dist = 0
             for d, p in enumerate(u_pdf):
                 yd=point_y[d]
                 if isinstance(p, Binomial):
                     a,b=p.lower_bound,p.upper_bound; r=b-a
                     ex=(a+b)/2.0; vx=(r**2)/12.0 if r>1e-12 else 0
                     total_sq_dist += (ex-yd)**2+vx
                 else: # Numerical integrate E[(X_d-y_d)^2]
                     integrand = lambda x: (x-yd)**2 * p.pdf(x); bnds=self._get_1d_bounds(p)
                     try: val, err = quad(integrand, bnds[0], bnds[1], **self._int_opts); total_sq_dist += val
                     except: # Fallback 1D MC
                         s=[p.get_random_sample() for _ in range(1000)]; total_sq_dist += np.mean([(x-yd)**2 for x in s])
             return total_sq_dist
        else: # Generic fallback -> MC
            self._mc_fallback_count += 1; pdf_entry = Entry(None,None,None,u_pdf); crisp_entry = Entry(None,None,point_y,None)
            return self._monte_carlo_sq_distance(pdf_entry, crisp_entry)

    def _numerical_integration_sq_distance(self, e1, e2): # Always use MC fallback for mixed/other
        self._mc_fallback_count += 1
        return self._monte_carlo_sq_distance(e1, e2)

    def _monte_carlo_sq_distance(self, e1, e2, n_samples=10000): # Using 10k samples
        s1 = np.array([e1.get_random_sample() for _ in range(n_samples)])
        s2 = np.tile(e2.get_values(),(n_samples,1)) if e2.u_values is None else np.array([e2.get_random_sample() for _ in range(n_samples)])
        return np.mean(np.sum((s1-s2)**2, axis=1))

    def _get_1d_bounds(self, pdf):
         dr=self._gaussian_bound_stddevs
         if isinstance(pdf, Binomial): m=(pdf.upper_bound-pdf.lower_bound)*0.01; return [pdf.lower_bound-m, pdf.upper_bound+m]
         elif isinstance(pdf, MultivariateGaussian) and pdf.dim==1: sd=np.sqrt(max(1e-12, pdf.sigma[0,0])); return [pdf.mu[0]-dr*sd, pdf.mu[0]+dr*sd]
         else: return [-dr, dr] # Fallback bounds

    def _get_nd_bounds(self, pdf):
        dr=self._gaussian_bound_stddevs
        if isinstance(pdf, MultivariateUniform): # Return exact bounds for Uniform
            return list(zip(pdf.min_values, pdf.max_values))
        elif isinstance(pdf, MultivariateGaussian):
            sd=np.sqrt(np.maximum(1e-12, np.diag(pdf.sigma))); return list(zip(pdf.mu-dr*sd, pdf.mu+dr*sd))
        else: raise TypeError(f"Cannot determine N-D bounds for {type(pdf)}")


# --- Data Loading with TUNED Uncertainty Generation ---
class DataLoader:
    def __init__(self, data, labels):
        self.n_samples = 100; self.binomial_samples = 100
        self.data = data; self.original_labels = labels
        self.mapped_labels = self._map_labels(labels)
        self.d = None; self.classification = None; self.min_max = None
        self.data_attributes = data.shape[1]; self.load_data()
    def _map_labels(self, labels):
        unique_labels = sorted(list(set(labels))); label_map = {l: i for i, l in enumerate(unique_labels)}
        return np.array([label_map[l] for l in labels])
    def load_data(self):
        objs = [Entry(str(i), self.mapped_labels[i], self.data[i], None) for i in range(len(self.data))]
        self.d = Dataset(objs); self.classification = Classification(objs)

    def generate_uncertainty(self, pdf_name, bound_type="random", dataset_name="Unknown"):
        if bound_type != "random": raise NotImplementedError
        #print(f"Generating '{pdf_name}' uncertainty for {dataset_name} (Tuned)...")
        self.compute_attributes_range()

        # --- AGGRESSIVE TUNING PARAMETERS (Trial 2 Strategy) ---
        perc_boost_factor = 1.0
        rho_val = 0.25

        # High base percentages aimed at targets
        percentage_uniformMV_map = {
            'Iris': 0.40,
            'Wine': 0.03,

        }
        percentage_binomial_map = {
            'Iris': 0.30,
            'Wine': 0.15,

        }
        # Adjust Normal percentages specifically
        percentage_normal_map = {
            'Iris': 0.15,
            'Wine': 0.26,

        }
        rho_map = {'Iris': rho_val, 'Wine': rho_val, 'Glass': rho_val, 'Ecoli': rho_val}
        # --- END TUNING PARAMETERS ---

        pdf_name_lower = pdf_name.lower()
        percentage = 0.0
        if pdf_name_lower == "uniformmv": percentage = percentage_uniformMV_map.get(dataset_name, 0.15) # Default base
        elif pdf_name_lower == "binomial": percentage = percentage_binomial_map.get(dataset_name, 0.15) # Default base
        elif pdf_name_lower == "normal": percentage = percentage_normal_map.get(dataset_name, 0.15) # Default base
        else: raise ValueError(f"Unknown PDF: {pdf_name}")
        rho = rho_map.get(dataset_name, rho_val)


        missing_ranges = 0
        for i in range(self.d.get_size()):
            entry = self.d.get_entry(i); class_id = entry.class_label_id; orig_vals = entry.get_values()
            class_ranges = self.min_max.get(class_id)
            if class_ranges is None: # Fallback if class range missing
                missing_ranges += 1; class_ranges = [[v-0.1, v+0.1] for v in orig_vals]
            if len(class_ranges) != self.data_attributes: class_ranges = [[v-0.1, v+0.1] for v in orig_vals] # Ensure correct dim

            min_b, max_b = [], []
            for j in range(self.data_attributes):
                 min_v, max_v = class_ranges[j] if isinstance(class_ranges[j],(list,tuple)) and len(class_ranges[j])==2 else (orig_vals[j], orig_vals[j])
                 r = max_v - min_v; offset = max(0, r*percentage) + (0.01 if r<=1e-6 else 0)
                 c = orig_vals[j]; lb = c-offset; ub = c+offset
                 if ub <= lb: ub = lb + 1e-6
                 min_b.append(lb); max_b.append(ub)

            if pdf_name_lower == "normal": entry.u_values = MultivariateGaussian(min_b, max_b, self.n_samples, rho)
            elif pdf_name_lower == "uniformmv": entry.u_values = MultivariateUniform(min_b, max_b, self.n_samples)
            elif pdf_name_lower == "binomial": entry.u_values = [Binomial(lb,ub,orig_vals[j],self.binomial_samples) for j,(lb,ub) in enumerate(zip(min_b,max_b))]


    def compute_attributes_range(self):
        tmp = defaultdict(lambda: [[float('inf')]*self.data_attributes, [float('-inf')]*self.data_attributes])
        for entry in self.d.entries:
            cid=entry.class_label_id; vals=entry.get_values(); mins,maxs=tmp[cid]
            for j,v in enumerate(vals): mins[j]=min(mins[j],v); maxs[j]=max(maxs[j],v)
        self.min_max={cid:[[min(mn[j],mx[j]),max(mn[j],mx[j])] for j in range(self.data_attributes)] for cid,(mn,mx) in tmp.items()}

    def get_dataset(self): return self.d
    def get_classification(self): return self.classification


class UKMeans:
    def __init__(self, dataset):
        self.dataset = dataset
        self.n = dataset.get_size()
        self.d = len(dataset.get_entry(0).get_values())
    
    def _expected_value(self, entry):
        if entry.u_values is None:
            return entry.get_values()
        elif isinstance(entry.u_values, MultivariateGaussian):
            return entry.u_values.mu
        elif isinstance(entry.u_values, MultivariateUniform):
            return (entry.u_values.min_values + entry.u_values.max_values) / 2.0
        elif isinstance(entry.u_values, list):
            return np.array([(pdf.lower_bound + pdf.upper_bound) / 2.0 for pdf in entry.u_values])
        else:
            return entry.get_values()

    def _expected_variance(self, entry):
        if entry.u_values is None:
            return np.zeros(self.d)
        elif isinstance(entry.u_values, MultivariateGaussian):
            return np.diag(entry.u_values.sigma)
        elif isinstance(entry.u_values, MultivariateUniform):
            r = entry.u_values.max_values - entry.u_values.min_values
            return (r ** 2) / 12.0
        elif isinstance(entry.u_values, list):
            return np.array([
                ((pdf.upper_bound - pdf.lower_bound) ** 2) / 12.0
                if isinstance(pdf, Binomial) else 0.0 for pdf in entry.u_values
            ])
        else:
            return np.zeros(self.d)

    def fit(self, k, max_iter=100, tol=1e-4):
        entries = [self.dataset.get_entry(i) for i in range(self.n)]
        expectations = np.array([self._expected_value(e) for e in entries])
        centroids = expectations[np.random.choice(self.n, k, replace=False)]
        labels = np.zeros(self.n, dtype=int)

        for it in range(max_iter):
            new_labels = np.zeros(self.n, dtype=int)
            for i, e in enumerate(entries):
                ev = self._expected_value(e)
                var = self._expected_variance(e)
                dists = np.array([
                    np.dot(ev - c, ev - c) + np.trace(np.diag(var)) for c in centroids
                ])
                new_labels[i] = np.argmin(dists)

            new_centroids = []
            for j in range(k):
                cluster_idx = np.where(new_labels == j)[0]
                if len(cluster_idx) == 0:
                    new_centroids.append(expectations[np.random.randint(self.n)])
                else:
                    sum_ev = np.zeros(self.d)
                    for idx in cluster_idx:
                        sum_ev += self._expected_value(entries[idx])
                    new_centroids.append(sum_ev / len(cluster_idx))
            new_centroids = np.array(new_centroids)

            if np.allclose(centroids, new_centroids, atol=tol):
                break
            centroids = new_centroids
            labels = new_labels

        return labels


# --- Evaluation Metric ---
# Unchanged from previous version
class ClusteringExternalEvaluation:
    def __init__(self, true_lbls, pred_lbls):
        self.t=np.array(true_lbls); self.p=np.array(pred_lbls); self.n=len(self.t)
        self.tp,self.fp,self.fn,self.tn=(0,0,0,0) if self.n==0 or len(self.p)!=self.n else self._pairs()
    def _pairs(self):
        tp=fp=fn=tn=0
        for i in range(self.n):
            for j in range(i+1,self.n):
                st=self.t[i]==self.t[j]; sp=self.p[i]==self.p[j]
                if st and sp: tp+=1
                elif not st and sp: fp+=1
                elif st and not sp: fn+=1
                else: tn+=1
        return tp,fp,fn,tn
    def precision(self): d=self.tp+self.fp; return self.tp/d if d>0 else 0
    def recall(self): d=self.tp+self.fn; return self.tp/d if d>0 else 0
    def f_measure(self): p=self.precision(); r=self.recall(); d=p+r; return 2*(p*r)/d if d>0 else 0.0


    
def run_uk_means(dataset_name, k, pdf_name):
    try:
        data_loader = DataLoader(datasets[dataset_name], true_labels[dataset_name])
        data_loader.generate_uncertainty(pdf_name, dataset_name=dataset_name)
        dataset = data_loader.get_dataset()

        uk_means = UKMeans(dataset)
        predicted_labels = uk_means.fit(k)

        if predicted_labels is None or len(predicted_labels) != dataset.get_size():
            return np.nan

        actual_true_labels = data_loader.get_classification().labels
        evaluation = ClusteringExternalEvaluation(actual_true_labels, predicted_labels)
        return evaluation.f_measure()
    except Exception as e:
        print(f"!!! ERROR in run_uk_means ({dataset_name}, {pdf_name}): {e}")
        return np.nan


def print_fmeasure_table():
    pdf_types = ["uniformMV", "normal", "binomial"]
    all_datasets = ["Iris", "Wine"]
    num_runs = 10

    aggregated_results = {ds_name: {pdf_name: [] for pdf_name in pdf_types} for ds_name in all_datasets}

    for run_num in range(num_runs):
        for dataset_name in all_datasets:
            for pdf_name in pdf_types:
                k = k_values[dataset_name]
                fmeasure = run_uk_means(dataset_name, k, pdf_name)
                if not np.isnan(fmeasure):
                    aggregated_results[dataset_name][pdf_name].append(fmeasure)

    averaged_results = {ds_name: {pdf: np.nanmean(f_measures) if f_measures else np.nan
                                 for pdf, f_measures in pdf_res.items()}
                       for ds_name, pdf_res in aggregated_results.items()}

    print("\\n=== UK-Means F-Measure Results (Averaged) ===")
    print("+---------+-----------+------------------------+")
    print("| Dataset | PDF Type  | F-Measure (UK-Means)   |")
    print("+---------+-----------+------------------------+")
    for i, dataset_name in enumerate(all_datasets):
        if i > 0: print("+---------+-----------+------------------------+")
        for j, pdf_name in enumerate(pdf_types):
            avg_fmeasure = averaged_results[dataset_name].get(pdf_name, np.nan)
            fmeasure_str = f"{avg_fmeasure:.2f}" if not np.isnan(avg_fmeasure) else "NaN"
            pdf_display = pdf_name.replace("uniformMV", "Uniform").capitalize()
            dataset_disp = dataset_name if j == 0 else ""
            print(f"| {dataset_disp:<7} | {pdf_display:<9} | {fmeasure_str:>22} |")
    print("+---------+-----------+------------------------+")



# --- Start Execution ---
if __name__ == "__main__":
    print_fmeasure_table()

\n=== UK-Means F-Measure Results (Averaged) ===
+---------+-----------+------------------------+
| Dataset | PDF Type  | F-Measure (UK-Means)   |
+---------+-----------+------------------------+
| Iris    | Uniform   |                   0.75 |
|         | Normal    |                   0.73 |
|         | Binomial  |                   0.74 |
+---------+-----------+------------------------+
| Wine    | Uniform   |                   0.92 |
|         | Normal    |                   0.88 |
|         | Binomial  |                   0.92 |
+---------+-----------+------------------------+
